# 1.1A - Persiapan Data: Lapisan Bronze (Raw Ingestion)

Notebook ini merupakan tahap pertama dalam pipeline **Retrieval-Augmented Generation (RAG)** menggunakan arsitektur **Medallion Lakehouse (Bronze -> Silver -> Gold)**.

### Ringkasan Proses:
1. **Setup Lingkungan Lab**: Menginisialisasi katalog, skema, dan konfigurasi dari `../Includes/Lab_Setup`.
2. **Definisi Sumber Data**: Menentukan lokasi folder dokumen mentah pada Databricks Unity Catalog Volume.
3. **Ekstraksi Dokumen Mentah (`binaryfile`)**: Membaca file dokumen mentah (PDF/dokumen teks) ke dalam DataFrame PySpark dalam format representasi biner dan menambahkan timestamp penyerapan (*ingestion time*).
4. **Penyimpanan ke Tabel Bronze**: Menyimpan data mentah apa adanya ke tabel Delta Lake `bronze_docs_raw` sebagai fondasi untuk tahap parsing dokumen di lapisan Silver.


### Langkah 1: Setup Lingkungan Lab
Menjalankan notebook setup untuk menginisialisasi variabel konfigurasi lab (seperti objek `course`, katalog, skema, dan volume di Unity Catalog).


In [0]:
%run ../Includes/Lab_Setup

### Langkah 2: Menentukan Lokasi Sumber Data
Menentukan path direktori tempat file dokumen mentah berada di dalam Databricks Unity Catalog Volume berdasarkan konfigurasi `course.dataset_volume`.


In [0]:
# Menentukan path direktori tempat file dokumen mentah disimpan pada Unity Catalog Volume
datasource = f'{course.dataset_volume}/docs'

# Menampilkan path direktori sumber data untuk verifikasi lokasi
print(datasource)

### Langkah 3: Membaca Dokumen Mentah sebagai Binary File
Membaca seluruh dokumen mentah secara paralel menggunakan format `binaryfile`. 
Format ini menyimpan konten file dalam representasi biner (`content`) bersama metadata file (`path`, `modificationTime`, `length`).
Kita juga menambahkan kolom `ingestion_time` untuk mencatat waktu data diserap ke dalam lakehouse.


In [0]:
# Mengimpor fungsi current_timestamp dari PySpark untuk mencatat waktu penyerapan data
from pyspark.sql.functions import current_timestamp

# Membaca seluruh file dokumen mentah dari direktori sumber data:
# - spark.read.format("binaryfile"): membaca file biner mentah (metadata path, length, modificationTime, dan byte content)
# - .load(datasource): memuat file dari path direktori volume yang sudah ditentukan
# - .withColumn("ingestion_time", current_timestamp()): menambahkan kolom waktu penyerapan (ingestion)
docs_raw_df = (spark.read.format("binaryfile")
                .load(datasource)
                .withColumn("ingestion_time", current_timestamp()))

# Menampilkan isi DataFrame dan mengurutkannya berdasarkan ukuran file (length) dari yang terkecil ke terbesar
display(docs_raw_df.orderBy("length"))

### Langkah 4: Menyimpan Data ke Tabel Delta Lapisan Bronze
Menyimpan DataFrame ke tabel Delta terkelola (*managed table*) bernama `bronze_docs_raw`.
Lapisan Bronze menyimpan data mentah apa adanya (*raw/unprocessed*) untuk auditabilitas dan kemudahan pemrosesan ulang di tahap berikutnya (lapisan Silver).


In [0]:
# Menyimpan DataFrame dokumen mentah ke tabel Delta Lake dengan nama `bronze_docs_raw`
# - mode("overwrite"): menimpa isi tabel jika sudah ada agar data selalu bersih dan terbarui
# - saveAsTable("bronze_docs_raw"): menyimpan ke skema Unity Catalog aktif sebagai tabel terkelola
docs_raw_df.write.mode("overwrite").saveAsTable("bronze_docs_raw")